# 전세대출 Top 5 상품 추천

**실행 순서**: 셀을 위에서 아래로 순서대로 실행하세요.  
**고객 변경**: Cell 3 의 `USER_ID` 값만 바꾸고 Cell 7(추천 실행) 만 다시 실행하면 됩니다.

### 필요 파일 (`/content/` 아래에 업로드)
| 파일 | 설명 |
|------|------|
| `jeonse_customer_profiles.csv` | 고객 개인·재무 정보 |
| `jeonse_property_gate.csv` | 매물·전세계약 정보 |
| `jeonse_product_catalog.csv` | 전세대출 상품 카탈로그 (A1~A6 + B001~) |

In [ ]:
# ──────────────────────────────────────────────────────────
# Cell 1 : 라이브러리 & 전역 상수
# ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

# ── 파일 경로
PROFILES_PATH = '/content/jeonse_customer_profiles.csv'
GATE_PATH     = '/content/jeonse_property_gate.csv'
CATALOG_PATH  = '/content/jeonse_product_catalog.csv'

# ── 스코어링 상수
LOAN_TERM_MONTHS = 24          # 전세대출 기본 만기 (개월)
GUARANTEE_SCORE  = {'HUG': 100, 'HF': 80, 'SGI': 60}

# ── 고객 유형별 가중치 (Approval, Interest, Limit, Afford, Guarantee, Preference)
WEIGHT_PROFILES = {
    'default':           (0.25, 0.25, 0.20, 0.15, 0.10, 0.05),
    'rate_sensitive':    (0.20, 0.38, 0.18, 0.14, 0.05, 0.05),
    'limit_constrained': (0.20, 0.15, 0.38, 0.15, 0.07, 0.05),
    'credit_vulnerable': (0.40, 0.20, 0.18, 0.12, 0.05, 0.05),
    'youth':             (0.20, 0.35, 0.20, 0.10, 0.10, 0.05),
    'newlywed_newborn':  (0.22, 0.30, 0.25, 0.13, 0.05, 0.05),
    'high_dsr_risk':     (0.25, 0.18, 0.15, 0.32, 0.05, 0.05),
}
WEIGHT_LABELS = {
    'default':           '기본형',
    'rate_sensitive':    '금리민감형',
    'limit_constrained': '한도부족형',
    'credit_vulnerable': '신용취약형',
    'youth':             '청년무주택형',
    'newlywed_newborn':  '신혼·신생아형',
    'high_dsr_risk':     '상환위험형',
}
print('✓ Cell 1 완료')

In [ ]:
# ──────────────────────────────────────────────────────────
# Cell 2 : 데이터 로드
# ──────────────────────────────────────────────────────────
profiles = pd.read_csv(PROFILES_PATH, dtype={'user_id': str})
gates    = pd.read_csv(GATE_PATH,     dtype={'user_id': str})
catalog  = pd.read_csv(CATALOG_PATH)

# 숫자 컬럼 강제 변환 (빈 셀 NaN 처리)
for col in ['base_rate_min', 'base_rate_max',
            'deposit_limit_capital', 'deposit_limit_non_capital',
            'max_loan_capital', 'max_loan_non_capital',
            'ltv_limit_pct', 'dsr_limit_pct', 'max_area_sqm',
            'credit_score_min', 'age_min', 'age_max',
            'income_limit_single', 'income_limit_couple', 'net_asset_limit']:
    if col in catalog.columns:
        catalog[col] = pd.to_numeric(catalog[col], errors='coerce')

# 금리 NaN 상품 제외 (스코어링 불가)
catalog = catalog.dropna(subset=['base_rate_min', 'base_rate_max']).reset_index(drop=True)

print(f'고객 프로필  : {len(profiles):>4d}명')
print(f'매물/대출 정보: {len(gates):>4d}건')
print(f'상품 카탈로그: {len(catalog):>4d}개  (정부 {len(catalog[catalog.product_id.str.startswith("A")])}개 + 금융권 {len(catalog[catalog.product_id.str.startswith("B")])}개)')
print(f'\n사용 가능한 고객 ID (처음 15개):')
print(', '.join(profiles['user_id'].tolist()[:15]))

In [ ]:
# ──────────────────────────────────────────────────────────
# Cell 3 : ★ 고객 ID 선택  ← 여기만 바꾸세요
# ──────────────────────────────────────────────────────────
USER_ID = 'U0001'   # ← 원하는 고객 ID 입력 (예: 'U0042', 'U0100', ...)
print(f'선택된 고객: {USER_ID}')

In [ ]:
# ──────────────────────────────────────────────────────────
# Cell 4 : Gate 평가 함수  (12개 조건 곱 연산)
# ──────────────────────────────────────────────────────────
def evaluate_gate(cust: pd.Series, gate: pd.Series, prod: pd.Series) -> dict:
    """
    반환: {조건명: 0/1, ..., 'pass': bool}
    단 하나라도 0이면 pass=False
    """
    is_capital = (gate['is_capital_area'] == 'Y')

    # ① 소득 조건: 기혼이면 부부합산, 미혼이면 본인
    if cust['marital_status'] == '기혼':
        income_ok = cust['combined_annual_income'] <= prod['income_limit_couple']
    else:
        income_ok = cust['annual_income'] <= prod['income_limit_single']

    # ② 보증금 한도
    dep_limit  = prod['deposit_limit_capital'] if is_capital else prod['deposit_limit_non_capital']
    deposit_ok = gate['deposit_amount'] <= dep_limit

    # ③ 주택 유형
    allowed    = str(prod['allowed_property_types']).split('|')
    prop_ok    = gate['property_type'] in allowed

    # ④ 상품별 필수 조건 (Conditional Gate)
    special_ok = True
    if str(prod.get('marriage_within_7y_required', 'N')) == 'Y':
        special_ok = (cust['marital_status'] == '기혼') and (cust['marriage_years'] <= 7)
    if str(prod.get('newborn_within_2y_required', 'N')) == 'Y':
        special_ok = special_ok and (cust['has_newborn_within_2y'] == 'Y')
    if str(prod.get('youth_only', 'N')) == 'Y':
        # age_min/age_max 이미 19~34로 설정돼 있어 연령 게이트와 중복이지만 명시적으로도 체크
        special_ok = special_ok and (19 <= cust['age'] <= 34)

    results = {
        '연령조건':     int(prod['age_min'] <= cust['age'] <= prod['age_max']),
        '무주택':       int(cust['is_homeless'] == 'Y'),
        '세대주':       int(cust['is_household_head'] == 'Y'),
        '소득조건':     int(bool(income_ok)),
        '순자산조건':   int(cust['net_asset'] <= prod['net_asset_limit']),
        '신용점수':     int(cust['credit_score_kcb'] >= prod['credit_score_min']),
        'DSR기준':      int(gate['dsr_estimated_pct'] <= prod['dsr_limit_pct']),
        '보증금한도':   int(bool(deposit_ok)),
        '주택유형':     int(bool(prop_ok)),
        '전용면적':     int(gate['exclusive_area_sqm'] <= prod['max_area_sqm']),
        '중복대출없음': int(cust['has_existing_jeonse_loan'] == 'N'),
        '정보완전성':   int(gate['info_completeness'] == 'Y'),
        '상품별조건':   int(special_ok),
    }
    results['pass'] = all(v == 1 for k, v in results.items() if k != 'pass')
    return results

print('✓ Cell 4 완료  (Gate 평가 함수)')

In [ ]:
# ──────────────────────────────────────────────────────────
# Cell 5 : 고객 유형 분류
# ──────────────────────────────────────────────────────────
def classify_customer(cust: pd.Series, gate: pd.Series, catalog: pd.DataFrame) -> str:
    """
    우선순위: 신용취약 > 상환위험 > 한도부족 > 청년 > 신혼·신생아 > 금리민감 > 기본
    """
    credit  = cust['credit_score_kcb']
    dsr     = gate['dsr_estimated_pct']
    income  = cust['combined_annual_income']
    age     = cust['age']
    loan_req = gate['loan_amount_requested']

    # 한도부족 기준: 요청 대출금이 전체 상품 max_loan 중앙값의 90% 초과
    median_limit = catalog['max_loan_capital'].median()

    if credit < 700:
        return 'credit_vulnerable'
    if dsr >= 35:
        return 'high_dsr_risk'
    if loan_req > median_limit * 0.9:
        return 'limit_constrained'
    if (19 <= age <= 34) and (cust['is_homeless'] == 'Y') and (cust['is_household_head'] == 'Y'):
        return 'youth'
    if (cust['marital_status'] == '기혼' and cust['marriage_years'] <= 7) or \
       (cust['has_newborn_within_2y'] == 'Y'):
        return 'newlywed_newborn'
    if income <= 4000:
        return 'rate_sensitive'
    return 'default'

print('✓ Cell 5 완료  (고객 유형 분류 함수)')

In [ ]:
# ──────────────────────────────────────────────────────────
# Cell 6 : 점수 계산 함수
# ──────────────────────────────────────────────────────────
def calc_monthly_payment(loan_man_won: float, annual_rate_pct: float,
                          months: int = 24) -> int:
    """분할상환(원리금균등) 기준 월 상환액 (만원 반환)"""
    loan = loan_man_won * 10000   # 원 단위
    r    = annual_rate_pct / 100 / 12
    if r > 0:
        monthly = loan * r * (1 + r) ** months / ((1 + r) ** months - 1)
    else:
        monthly = loan / months
    return max(1, int(monthly / 10000))  # 만원 단위 반환


def score_product(cust: pd.Series, gate: pd.Series, prod: pd.Series,
                  rate_range: tuple, profile_key: str) -> float:
    """
    JeonseLoanRecScore =
        w1*ApprovalScore + w2*InterestBenefitScore + w3*LimitCoverageScore
      + w4*AffordabilityScore + w5*GuaranteeStabilityScore + w6*PreferenceMatchScore
      - RiskPenalty
    """
    weights    = WEIGHT_PROFILES[profile_key]
    is_capital = (gate['is_capital_area'] == 'Y')

    # 대표 금리 (최저+최고 평균)
    rep_rate = (prod['base_rate_min'] + prod['base_rate_max']) / 2

    # ── 1. ApprovalScore  (신용 여유 60% + 소득 여유 40%)
    credit_margin = max(0.0,
        (cust['credit_score_kcb'] - prod['credit_score_min']) /
        max(1, 1000 - prod['credit_score_min'])
    ) * 100

    income_lim = prod['income_limit_couple'] if cust['marital_status'] == '기혼' \
                 else prod['income_limit_single']
    if income_lim < 99999:
        income_margin = max(0.0, (income_lim - cust['combined_annual_income']) /
                           max(1, income_lim)) * 100
    else:
        income_margin = 100.0
    approval_score = 0.6 * credit_margin + 0.4 * income_margin

    # ── 2. InterestBenefitScore  (후보군 내 금리 정규화)
    min_rate, max_rate = rate_range
    if max_rate > min_rate:
        interest_score = max(0.0, (max_rate - rep_rate) / (max_rate - min_rate) * 100)
    else:
        interest_score = 50.0

    # ── 3. LimitCoverageScore  (LTV 캡 적용 실질 한도 / 신청 대출금)
    max_loan  = prod['max_loan_capital'] if is_capital else prod['max_loan_non_capital']
    ltv_cap   = gate['deposit_amount'] * prod['ltv_limit_pct'] / 100
    eff_limit = min(max_loan, ltv_cap)
    loan_req  = gate['loan_amount_requested']
    limit_score = min(100.0, eff_limit / max(1, loan_req) * 100)

    # ── 4. AffordabilityScore  (DSR 여유도)
    dsr_limit   = prod['dsr_limit_pct']
    dsr_current = gate['dsr_estimated_pct']
    affordability_score = max(0.0, (dsr_limit - dsr_current) / dsr_limit * 100)

    # ── 5. GuaranteeStabilityScore
    guarantee_score = float(GUARANTEE_SCORE.get(str(prod['guarantee_agency']), 60))

    # ── 6. PreferenceMatchScore  (금리방식 선호 일치 여부)
    notes_str = str(prod.get('notes', ''))
    pref_map  = {'고정': '고정금리', '변동': '변동금리', '혼합': '혼합금리'}
    pref_key  = pref_map.get(str(gate.get('interest_rate_type', '')), '')
    if pref_key and pref_key in notes_str:
        pref_score = 100.0
    elif pref_key:
        pref_score = 0.0
    else:
        pref_score = 50.0

    # ── RiskPenalty
    penalty = 0.0
    if 30 <= dsr_current < 40:
        penalty += 10
    if 600 <= cust['credit_score_kcb'] < 700:
        penalty += 15
    if gate['deposit_to_market_ratio_pct'] > 90:
        penalty += 20
    if gate['senior_plus_deposit_to_market_pct'] > 80:
        penalty += 10

    w = weights
    score = (
        w[0] * approval_score    +
        w[1] * interest_score    +
        w[2] * limit_score       +
        w[3] * affordability_score +
        w[4] * guarantee_score   +
        w[5] * pref_score
        - penalty
    )
    return max(0.0, round(score, 2))

print('✓ Cell 6 완료  (점수 계산 함수)')

In [ ]:
# ──────────────────────────────────────────────────────────
# Cell 7 : 추천 실행 & 결과 출력
# ──────────────────────────────────────────────────────────
def get_join_label(join_way) -> str:
    """가입방법 → 심사 레이블"""
    if pd.isna(join_way) or str(join_way).strip() == '':
        return '영업점'
    jw = str(join_way)
    has_online  = any(k in jw for k in ['스마트폰', '인터넷', '모바일', '앱'])
    has_offline = '영업점' in jw or '모집인' in jw
    if has_online and not has_offline:
        return '비대면'
    if has_online and has_offline:
        return '비대면+영업점'
    return '영업점'


def recommend(user_id: str, top_n: int = 5):
    # ── 데이터 조회
    cust_rows = profiles[profiles['user_id'] == user_id]
    gate_rows = gates[gates['user_id'] == user_id]

    if cust_rows.empty:
        print(f"❌ 고객 ID '{user_id}' 를 찾을 수 없습니다.")
        print(f"   사용 가능 예시: {', '.join(profiles['user_id'].tolist()[:5])}")
        return
    if gate_rows.empty:
        print(f"❌ '{user_id}' 의 매물 정보(jeonse_property_gate.csv)가 없습니다.")
        return

    cust = cust_rows.iloc[0]
    gate = gate_rows.iloc[0]

    # ── 고객 유형 분류
    profile_key   = classify_customer(cust, gate, catalog)
    profile_label = WEIGHT_LABELS[profile_key]

    # ── Gate 평가
    passed_products = []
    fail_log        = []

    for _, prod in catalog.iterrows():
        g = evaluate_gate(cust, gate, prod)
        if g['pass']:
            passed_products.append(prod)
        else:
            failed_keys = [k for k, v in g.items() if k != 'pass' and v == 0]
            fail_log.append((prod['product_id'], prod['product_name'], failed_keys))

    if not passed_products:
        print("❌ Gate를 통과한 상품이 없습니다. 고객 조건을 확인하세요.")
        return

    # ── 후보군 금리 범위 (InterestBenefitScore 정규화)
    rep_rates  = [(p['base_rate_min'] + p['base_rate_max']) / 2 for p in passed_products]
    rate_range = (min(rep_rates), max(rep_rates))

    # ── 점수 계산 & 정렬
    scored = sorted(
        [(score_product(cust, gate, p, rate_range, profile_key), p) for p in passed_products],
        key=lambda x: -x[0]
    )
    top_list = scored[:top_n]

    # ────────────────────────────────────────────────────
    # HTML 렌더링
    # ────────────────────────────────────────────────────
    monthly_income = int(cust['annual_income'] * 10000 / 12)
    is_capital     = (gate['is_capital_area'] == 'Y')
    loan_req       = int(gate['loan_amount_requested'])
    deposit        = int(gate['deposit_amount'])
    delinquent     = '없음' if cust['has_existing_jeonse_loan'] == 'N' else '있음'

    # ─ 왼쪽 고객 정보 카드
    left_card = f"""
    <div style="background:#1e2a3a;color:#dce8f5;padding:22px 18px;border-radius:10px;
                min-width:220px;max-width:240px;font-size:13px;line-height:2.0;
                box-shadow:2px 4px 12px rgba(0,0,0,0.25);">
      <div style="font-size:11px;color:#7a9bb8;margin-bottom:8px;">고객 {cust['user_id']}</div>
      <div>월 소득 &nbsp;<b style="color:#fff;">{monthly_income:,}원</b></div>
      <div>기존 부채 &nbsp;<b style="color:#fff;">{int(cust['total_existing_debt']):,}만원</b></div>
      <div>희망 대출 &nbsp;<b style="color:#fff;">{loan_req:,}만원</b></div>
      <div>전세 보증금 <b style="color:#fff;">{deposit:,}만원</b></div>
      <div>예상 DSR &nbsp;<b style="color:#{'f5a623' if gate['dsr_estimated_pct']>=30 else 'fff'};">{gate['dsr_estimated_pct']}%</b></div>
      <div>신용점수 &nbsp;<b style="color:#{'f87171' if cust['credit_score_kcb']<700 else 'fff'};">{int(cust['credit_score_kcb'])}점</b></div>
      <div>최근 연체 &nbsp;<b style="color:#fff;">{delinquent}</b></div>
      <hr style="border-color:#2e4060;margin:10px 0">
      <div style="font-size:11px;color:#7a9bb8;">고객 유형</div>
      <div style="color:#f5c842;font-weight:bold;font-size:14px;">{profile_label}</div>
      <div style="font-size:10px;color:#5a7a9a;margin-top:4px;line-height:1.4;">
        가중치: 승인{WEIGHT_PROFILES[profile_key][0]:.0%} 금리{WEIGHT_PROFILES[profile_key][1]:.0%}
        한도{WEIGHT_PROFILES[profile_key][2]:.0%}<br>상환{WEIGHT_PROFILES[profile_key][3]:.0%}
        보증{WEIGHT_PROFILES[profile_key][4]:.0%} 우대{WEIGHT_PROFILES[profile_key][5]:.0%}
      </div>
    </div>
    """

    # ─ 상품 테이블 행 생성
    rows_html = ''
    medal = ['🥇', '🥈', '🥉', '4', '5']

    for rank, (score, prod) in enumerate(top_list, 0):
        max_loan   = prod['max_loan_capital'] if is_capital else prod['max_loan_non_capital']
        ltv_cap    = gate['deposit_amount'] * prod['ltv_limit_pct'] / 100
        eff_limit  = int(min(max_loan, ltv_cap))
        actual_loan = min(loan_req, eff_limit)

        rep_rate   = (prod['base_rate_min'] + prod['base_rate_max']) / 2
        monthly_pm = calc_monthly_payment(actual_loan, rep_rate)

        rate_str    = f"{prod['base_rate_min']:.2f}%"
        rate_clr    = '#dc2626' if prod['base_rate_min'] > 8 else ('#059669' if prod['base_rate_min'] < 3.5 else '#1e293b')
        limit_clr   = '#dc2626' if eff_limit < loan_req else '#1e293b'
        join_label  = get_join_label(prod.get('join_way', ''))

        # 상품명 축약
        pname = str(prod['product_name'])
        if len(pname) > 24:
            pname = pname[:23] + '…'

        bg = '#f8fafc' if rank % 2 == 0 else '#ffffff'
        bold = 'font-weight:600;' if rank == 0 else ''

        rows_html += f"""
        <tr style="background:{bg};border-bottom:1px solid #e5eaf0;{bold}">
          <td style="padding:10px 14px;max-width:260px;">
            {medal[rank]}&nbsp;{pname}
            <span style="font-size:10px;color:#888;margin-left:6px;">[{prod['guarantee_agency']}]</span>
          </td>
          <td style="padding:10px 14px;text-align:center;color:{rate_clr};font-weight:600;">{rate_str}</td>
          <td style="padding:10px 14px;text-align:center;color:{limit_clr};">{eff_limit:,}만원</td>
          <td style="padding:10px 14px;text-align:center;">{monthly_pm:,}만원</td>
          <td style="padding:10px 14px;text-align:center;font-size:12px;color:#555;">{join_label}</td>
          <td style="padding:10px 14px;text-align:center;font-size:12px;
                     color:#1e6bb8;font-weight:600;">{score:.1f}점</td>
        </tr>
        """

    # ─ 최종 HTML 조립
    html = f"""
    <style>
      .rec-table th {{
        background:#1e2a3a;color:#fff;padding:10px 14px;
        font-size:13px;font-weight:500;
      }}
      .rec-table td {{ font-size:13px; }}
    </style>
    <div style="font-family:'Apple SD Gothic Neo','Malgun Gothic',sans-serif;margin:16px 0;">
      <div style="font-size:12px;color:#94a3b8;margin-bottom:4px;">대출 트랙 예제</div>
      <h2 style="margin:0 0 16px 0;font-size:20px;color:#0f172a;">
        고객 {user_id}와 후보 상품
      </h2>
      <div style="display:flex;gap:20px;align-items:flex-start;">
        {left_card}
        <div style="flex:1;">
          <div style="font-size:12px;color:#64748b;margin-bottom:8px;">
            Gate 통과 <b>{len(passed_products)}</b>개 상품 중 상위 <b>{len(top_list)}</b>개 표시
            &nbsp;|&nbsp; 가중치 프로파일:&nbsp;
            <b style="color:#1e6bb8;">{profile_label}</b>
          </div>
          <table class="rec-table" style="width:100%;border-collapse:collapse;
                         box-shadow:0 1px 6px rgba(0,0,0,0.08);border-radius:8px;
                         overflow:hidden;">
            <thead>
              <tr>
                <th style="text-align:left;">상품</th>
                <th>금리(최저)</th>
                <th>실질 한도</th>
                <th>월상환 (24M)</th>
                <th>심사</th>
                <th>추천점수</th>
              </tr>
            </thead>
            <tbody>{rows_html}</tbody>
          </table>
          <div style="margin-top:8px;font-size:11px;color:#94a3b8;line-height:1.6;">
            * 실질 한도: LTV({top_list[0][1]['ltv_limit_pct']:.0f}%) 및 지역 상한 적용 결과 &nbsp;|
            월상환: 분할상환(원리금균등) 24개월 기준 추정치 &nbsp;|
            금리: 상품 기준금리 최저값 기준 (우대금리 미반영)
          </div>
        </div>
      </div>
    </div>
    """
    display(HTML(html))

    # ── 텍스트 요약 (터미널 / 로그용)
    print(f"\n{'='*65}")
    print(f"  고객 {user_id} | 유형: {profile_label} | Gate 통과 상품: {len(passed_products)}개")
    print(f"{'='*65}")
    for rank, (score, prod) in enumerate(top_list, 1):
        max_loan  = prod['max_loan_capital'] if is_capital else prod['max_loan_non_capital']
        ltv_cap   = gate['deposit_amount'] * prod['ltv_limit_pct'] / 100
        eff_limit = int(min(max_loan, ltv_cap))
        print(f"  {rank}위 [{score:5.1f}점]  {prod['product_name']}")
        print(f"         금리: {prod['base_rate_min']}~{prod['base_rate_max']}%  "
              f"한도: {eff_limit:,}만원  보증: {prod['guarantee_agency']}")
    print(f"{'='*65}")

    # ── Gate 실패 상품 요약 (처음 5개)
    if fail_log:
        print(f"\n[Gate 탈락 상품 — 처음 5개]")
        for pid, pname, reasons in fail_log[:5]:
            short = pname if len(pname) <= 26 else pname[:25] + '…'
            print(f"  ✗ [{pid}] {short}  →  {', '.join(reasons)}")


# ─────────────────── 실행 ───────────────────
recommend(USER_ID, top_n=5)